In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  01_coleta.ipynb — Coleta de dados SI-PNI via PySUS         ║
# ║  TCC: Cobertura Vacinal no Piauí (2015–2023)                ║
# ╚══════════════════════════════════════════════════════════════╝

# ── CÉLULA 1: Instalar PySUS ──────────────────────────────────
# ⚠️ Após rodar esta célula, vá em:
#    Ambiente de execução > Reiniciar ambiente
#    Depois continue a partir da Célula 2

!pip install git+https://github.com/AlertaDengue/PySUS.git --upgrade

  Cloning https://github.com/AlertaDengue/PySUS.git to /tmp/pip-req-build-cc_71oce
  Running command git clone --filter=blob:none --quiet https://github.com/AlertaDengue/PySUS.git /tmp/pip-req-build-cc_71oce
  Resolved https://github.com/AlertaDengue/PySUS.git to commit c4c643dea4c992daf0fe54bffe3684cf5c6d1138
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
# ── CÉLULA 2: Montar Google Drive ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

RAIZ           = '/content/drive/MyDrive/TCC_Vacinal_Piaui'
DADOS_BRUTOS   = f'{RAIZ}/dados_brutos'
DADOS_TRATADOS = f'{RAIZ}/dados_tratados'

for pasta in [DADOS_BRUTOS, DADOS_TRATADOS]:
    os.makedirs(pasta, exist_ok=True)

print("✅ Drive montado e pastas prontas!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado e pastas prontas!


In [9]:
# ── CÉLULA DE DIAGNÓSTICO — rode antes de qualquer coisa ──────
import pysus
print("Versão:", pysus.__version__)

# Lista todos os módulos disponíveis
import pkgutil
mods = [m.name for m in pkgutil.iter_modules(pysus.__path__)]
print("Módulos disponíveis:", mods)

Versão: 2.3.0
Módulos disponíveis: ['api', 'cli', 'tui', 'utils']


In [13]:
# ── CÉLULA 3: Download com a API nova do PySUS 2.3.0 ──────────
from pysus import pni
import pandas as pd

ANOS = list(range(2015, 2024))
frames = []

for ano in ANOS:
    print(f"⬇️  Baixando PI — {ano}...", end=" ")
    try:
        # The pni() function returns a list of file paths in PySUS 2.3.0
        file_paths = pni(state="PI", year=ano)
        if file_paths:
            # Read each parquet file into a DataFrame and concatenate them
            df_ano = pd.concat([pd.read_parquet(f) for f in file_paths], ignore_index=True)
            df_ano['ano'] = ano
            frames.append(df_ano)
            print(f"✅ {len(df_ano):,} linhas")
        else:
            print(f"⚠️  Nenhum arquivo baixado para PI — {ano}")
    except Exception as e:
        print(f"⚠️  Erro: {e}")

df_bruto = pd.concat(frames, ignore_index=True)
print(f"\n📦 {df_bruto.shape[0]:,} linhas × {df_bruto.shape[1]} colunas")
print(df_bruto.columns.tolist())

⬇️  Baixando PI — 2015... 

✅ 203,347 linhas
⬇️  Baixando PI — 2016... 

✅ 214,275 linhas
⬇️  Baixando PI — 2017... 

✅ 199,616 linhas
⬇️  Baixando PI — 2018... 

✅ 214,562 linhas
⬇️  Baixando PI — 2019... 

✅ 61,590 linhas
⬇️  Baixando PI — 2020... ⚠️  Nenhum arquivo baixado para PI — 2020
⬇️  Baixando PI — 2021... ⚠️  Nenhum arquivo baixado para PI — 2021
⬇️  Baixando PI — 2022... ⚠️  Nenhum arquivo baixado para PI — 2022
⬇️  Baixando PI — 2023... ⚠️  Nenhum arquivo baixado para PI — 2023

📦 893,390 linhas × 15 colunas
['ANO', 'UF', 'MUNIC', 'IMUNO', 'QT_DOSE', 'POP', 'COBERT', 'ANOMES', 'MES', 'FX_ETARIA', 'DOSE', 'DOSE1', 'DOSEN', 'DIFER', 'ano']


In [14]:
# ── CÉLULA 4: Inspecionar os dados recebidos ──────────────────

print("=== COLUNAS E TIPOS ===")
print(df_bruto.dtypes)

print("\n=== ANOS DISPONÍVEIS ===")
print(df_bruto['ANO'].value_counts().sort_index())

print("\n=== VACINAS DISPONÍVEIS (IMUNO) ===")
print(df_bruto['IMUNO'].value_counts())

print("\n=== FAIXAS ETÁRIAS ===")
print(df_bruto['FX_ETARIA'].value_counts())

print("\n=== AMOSTRA — Parnaíba ===")
parnaiba = df_bruto[df_bruto['MUNIC'].astype(str).str.contains('2207702|Parnaíba|PARNAIBA', case=False, na=False)]
print(f"Linhas de Parnaíba: {len(parnaiba)}")
print(parnaiba.head(5).to_string())

print("\n=== COBERTURA — estatísticas ===")
print(df_bruto['COBERT'].describe())

=== COLUNAS E TIPOS ===
ANO          object
UF           object
MUNIC        object
IMUNO        object
QT_DOSE      object
POP          object
COBERT       object
ANOMES       object
MES          object
FX_ETARIA    object
DOSE         object
DOSE1        object
DOSEN        object
DIFER        object
ano           int64
dtype: object

=== ANOS DISPONÍVEIS ===
ANO
2015    203347
2016    214275
2017    199616
2018    214562
2019     61590
Name: count, dtype: int64

=== VACINAS DISPONÍVEIS (IMUNO) ===
IMUNO
08     107158
04      69859
06      58332
53      54971
21      53604
        ...  
07          2
91          2
100         2
32          2
60          1
Name: count, Length: 125, dtype: int64

=== FAIXAS ETÁRIAS ===
FX_ETARIA
50    189910
51    118848
61     44087
G5     27284
F5     27074
       ...  
36         3
76         2
A4         2
A8         1
06         1
Name: count, Length: 134, dtype: int64

=== AMOSTRA — Parnaíba ===
Linhas de Parnaíba: 0
Empty DataFrame
Columns: [ANO

* Todas as colunas são object — os dados vieram como string, precisam de conversão
* IMUNO são códigos numéricos (08, 04, 06...) — precisamos de uma tabela de-para para saber qual vacina é qual
* Parnaíba não encontrada — o campo MUNIC provavelmente usa código IBGE (2207702), não nome
* COBERT tem só 30.265 valores num dataset de 893.390 linhas — a maioria é nula, e os valores são string

In [15]:
# ── CÉLULA 5: Salvar os dados de 2015–2019 no Drive ───────────

caminho = f'{DADOS_BRUTOS}/pni_pi_2015_2019_bruto.parquet'
df_bruto.to_parquet(caminho, index=False)
print(f"✅ Salvo: {caminho}")
print(f"   Tamanho: {os.path.getsize(caminho)/1024:.0f} KB")

✅ Salvo: /content/drive/MyDrive/TCC_Vacinal_Piaui/dados_brutos/pni_pi_2015_2019_bruto.parquet
   Tamanho: 2747 KB


In [16]:
# ── CÉLULA 6: Resolver o gap 2020–2023 via OpenDataSUS ────────
# O SI-PNI mudou em 2020 — os dados estão no OpenDataSUS como
# microdados individualizados de doses aplicadas (muito maiores)
# Para cobertura % por município em 2020–2023, o TabNet é a
# fonte mais direta e confiável

# Vamos checar o que o PySUS consegue via anos alternativos
from pysus import pni

print("Testando anos alternativos...")
for ano in [2020, 2021, 2022, 2023]:
    try:
        files = pni(state="PI", year=ano)
        print(f"  {ano}: {len(files) if files else 0} arquivo(s)")
    except Exception as e:
        print(f"  {ano}: erro — {e}")

Testando anos alternativos...
  2020: 0 arquivo(s)
  2021: 0 arquivo(s)
  2022: 0 arquivo(s)
  2023: 0 arquivo(s)


In [17]:
# ── CÉLULA 7: Diagnóstico dos campos chave ────────────────────

# Ver como MUNIC está armazenado
print("=== MUNIC — primeiros valores únicos ===")
print(df_bruto['MUNIC'].dropna().unique()[:20])

# Ver como COBERT está armazenado
print("\n=== COBERT — primeiros valores não nulos ===")
print(df_bruto[df_bruto['COBERT'].notna() & (df_bruto['COBERT'] != '')]['COBERT'].head(20).tolist())

# Ver como QT_DOSE e POP estão
print("\n=== QT_DOSE — amostra ===")
print(df_bruto['QT_DOSE'].dropna().head(10).tolist())

print("\n=== POP — amostra ===")
print(df_bruto['POP'].dropna().head(10).tolist())

# Combinações de IMUNO + FX_ETARIA mais frequentes
print("\n=== Top 15 combinações IMUNO + FX_ETARIA ===")
print(df_bruto.groupby(['IMUNO','FX_ETARIA']).size().sort_values(ascending=False).head(15))

=== MUNIC — primeiros valores únicos ===
['220005' '220010' '220020' '220025' '220027' '220030' '220040' '220045'
 '220050' '220060' '220070' '220080' '220090' '220095' '220100' '220105'
 '220110' '220115' '220117' '220120']

=== COBERT — primeiros valores não nulos ===
['103,75', '27,5', '73,75', '45', '0', '0', '0', '0', '0', '80', '111,25', '98,75', '73,75', '0', '91,25', '0', '0', '0', '0', '0']

=== QT_DOSE — amostra ===
['83', '22', '59', '36', '53', '0', '0', '0', '0', '64']

=== POP — amostra ===
['80', '80', '80', '80', '', '', '', '', '', '80']

=== Top 15 combinações IMUNO + FX_ETARIA ===
IMUNO  FX_ETARIA
04     61           34622
52     50           32037
49     50           29196
12     50           25194
53     50           21763
21     51           17487
08     S6           13176
       S5           12883
93     C4           12478
04     01           12435
       50           12270
61     74           11811
19     51           11780
53     51           11678
12     51   

In [18]:
# ── CÉLULA 8: Tabela de-para — códigos de vacinas do SI-PNI ───
# Fonte: dicionário oficial do SI-PNI (DATASUS)

VACINAS = {
    '04': 'DTP',
    '06': 'BCG',
    '07': 'Febre Amarela',
    '08': 'Poliomielite',
    '09': 'Sarampo',
    '16': 'Hepatite B',
    '21': 'Tríplice Viral (SCR)',
    '22': 'Dupla Adulto (dT)',
    '23': 'Hepatite A',
    '25': 'Meningocócica C',
    '28': 'Pentavalente (DTP+Hib+HepB)',
    '36': 'Rotavírus',
    '40': 'Pneumocócica 10V',
    '53': 'Influenza',
    '55': 'HPV',
    '56': 'VIP (Polio Inativada)',
    '60': 'Varicela',
    '87': 'BCG (dose)',
    '91': 'COVID-19',
    '100': 'Dengue',
}

FAIXAS = {
    '50': '< 1 ano',
    '51': '1 ano',
    '52': '2 anos',
    '53': '3 anos',
    '54': '4 anos',
    '61': 'Adolescente',
    'F5': 'Feminino < 1 ano',
    'G5': 'Masculino < 1 ano',
}

# Aplicar mapeamentos
df_bruto['vacina_nome'] = df_bruto['IMUNO'].map(VACINAS).fillna('Código ' + df_bruto['IMUNO'].astype(str))
df_bruto['faixa_nome']  = df_bruto['FX_ETARIA'].map(FAIXAS).fillna(df_bruto['FX_ETARIA'])

print("=== Vacinas mapeadas ===")
print(df_bruto['vacina_nome'].value_counts().head(15))

=== Vacinas mapeadas ===
vacina_nome
Poliomielite            107158
DTP                      69859
BCG                      58332
Influenza                54971
Tríplice Viral (SCR)     53604
Código 52                40187
Código 93                39931
Código 12                39319
Código 19                37641
Código 61                37070
Código 49                32978
Hepatite B               32004
Código                   25588
Código 94                21910
Código 14                19398
Name: count, dtype: int64


In [19]:
# ── CÉLULA 9: Converter tipos e criar coluna município ─────────

# Código IBGE do Piauí: municípios começam com 22
# Parnaíba = 2207702
print("=== Valores únicos de MUNIC (amostra) ===")
print(sorted(df_bruto['MUNIC'].dropna().unique())[:30])

# Converter colunas numéricas
for col in ['QT_DOSE', 'POP', 'COBERT', 'ANO', 'MES']:
    df_bruto[col] = pd.to_numeric(df_bruto[col], errors='coerce')

# Calcular cobertura onde estiver nula (doses/pop * 100)
mask = df_bruto['COBERT'].isna() & df_bruto['QT_DOSE'].notna() & df_bruto['POP'].notna() & (df_bruto['POP'] > 0)
df_bruto.loc[mask, 'COBERT'] = (df_bruto.loc[mask, 'QT_DOSE'] / df_bruto.loc[mask, 'POP'] * 100).round(2)

print(f"\nCOBERT preenchida: {df_bruto['COBERT'].notna().sum():,} de {len(df_bruto):,} linhas")
print(df_bruto['COBERT'].describe())

=== Valores únicos de MUNIC (amostra) ===
['220005', '220010', '220020', '220025', '220027', '220030', '220040', '220045', '220050', '220060', '220070', '220080', '220090', '220095', '220100', '220105', '220110', '220115', '220117', '220120', '220130', '220140', '220150', '220155', '220157', '220160', '220170', '220173', '220177', '220180']

COBERT preenchida: 30,265 de 893,390 linhas
count    30265.000000
mean        50.872089
std         43.515211
min          0.000000
25%          9.300000
50%         45.100000
75%         85.470000
max        342.860000
Name: COBERT, dtype: float64


In [20]:
# ── CÉLULA 10: Filtrar Parnaíba após diagnóstico ──────────────
# Ajuste o valor de MUNIC_PARNAIBA conforme o resultado da célula 9

MUNIC_PARNAIBA = '220770'   # código IBGE — pode ser 6 ou 7 dígitos, veja o output

parnaiba = df_bruto[df_bruto['MUNIC'].astype(str).str.startswith(MUNIC_PARNAIBA)]
print(f"Linhas de Parnaíba: {len(parnaiba)}")
print(parnaiba[['ANO','vacina_nome','faixa_nome','QT_DOSE','POP','COBERT']].head(10).to_string())

Linhas de Parnaíba: 14867
       ANO vacina_nome faixa_nome  QT_DOSE     POP  COBERT
5026  2015  Código 000        NaN     1594  2280.0   69.91
5027  2015  Código 003        NaN      814  2280.0   35.70
5028  2015  Código 006        NaN     1892  2280.0   82.98
5029  2015  Código 010        NaN     1466  2280.0   64.30
5030  2015  Código 101        NaN     1276     NaN    0.00
5031  2015  Código 102        NaN      196     NaN    0.00
5032  2015  Código 108        NaN        0     NaN    0.00
5033  2015  Código 109        NaN        0     NaN    0.00
5034  2015  Código 110        NaN        0     NaN    0.00
5035  2015  Código 111        NaN        0     NaN    0.00


In [21]:
# ── CÉLULA 11: Corrigir COBERT (vírgula → ponto) ──────────────

df_bruto['COBERT'] = (
    df_bruto['COBERT']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df_bruto['COBERT'] = pd.to_numeric(df_bruto['COBERT'], errors='coerce')

print("COBERT corrigida:")
print(df_bruto['COBERT'].describe())
print(f"\nNulos: {df_bruto['COBERT'].isna().sum():,}")

COBERT corrigida:
count    30265.000000
mean        50.872089
std         43.515211
min          0.000000
25%          9.300000
50%         45.100000
75%         85.470000
max        342.860000
Name: COBERT, dtype: float64

Nulos: 863,125


In [22]:
# ── Raspar nomes de vacinas direto do TabNet ──────────────────
import requests
from bs4 import BeautifulSoup

url = "http://tabnet.datasus.gov.br/cgi/dhdat.exe?bd_pni/cpniuf.def"
resp = requests.get(url, timeout=15)
resp.encoding = "latin-1"

soup = BeautifulSoup(resp.text, "html.parser")

# Os imunobiológicos ficam num <select> específico
select = soup.find("select", {"name": "Incremento"})
if select:
    opcoes = [(opt["value"], opt.text.strip()) for opt in select.find_all("option")]
    df_vacinas = pd.DataFrame(opcoes, columns=["codigo", "nome"])
    print(df_vacinas)
    df_vacinas.to_csv(f'{DADOS_BRUTOS}/tabela_vacinas.csv', index=False)
else:
    print("Select não encontrado — o TabNet pode ter mudado o layout")

/usr/local/lib/python3.12/dist-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Select não encontrado — o TabNet pode ter mudado o layout


In [23]:
# ── CÉLULA 12: Dicionário completo de vacinas SI-PNI ──────────
# Fonte: Tabela IMUNO do dicionário de dados do PNI/DATASUS
# Inclui códigos de 2 e 3 dígitos (alguns municípios usam zero-padded)

VACINAS_COMPLETO = {
    # Códigos sem zero à esquerda E com zero à esquerda (ambos os formatos)
    '4':   'DTP',                        '04':  'DTP',
    '6':   'BCG',                        '06':  'BCG',
    '7':   'Febre Amarela',              '07':  'Febre Amarela',
    '8':   'Poliomielite (VOP)',         '08':  'Poliomielite (VOP)',
    '9':   'Sarampo',                    '09':  'Sarampo',
    '12':  'Hepatite B',
    '13':  'Tríplice Bacteriana (DTP)',
    '14':  'Dupla Infantil (DT)',
    '16':  'Hepatite B (adulto)',
    '19':  'BCG (revacinação)',
    '21':  'Tríplice Viral (SCR)',
    '22':  'Dupla Adulto (dT)',
    '23':  'Hepatite A',
    '25':  'Meningocócica C (conjugada)',
    '27':  'Pneumocócica 23V',
    '28':  'Pentavalente (DTP+Hib+HepB)',
    '36':  'Rotavírus Humano (VORH)',
    '38':  'Influenza (campanha)',
    '40':  'Pneumocócica 10V',
    '49':  'DTP (reforço)',
    '52':  'Poliomielite (VIP)',
    '53':  'Influenza',
    '55':  'HPV Quadrivalente',
    '56':  'VIP (Polio Inativada)',
    '60':  'Varicela',
    '61':  'Meningocócica ACWY',
    '87':  'BCG',
    '91':  'COVID-19',
    '93':  'Pneumocócica 10V (reforço)',
    '94':  'Meningocócica C (reforço)',
    '100': 'Dengue',
    # Códigos de 3 dígitos zerados (formato Parnaíba)
    '000': 'Hepatite B (ao nascer)',
    '003': 'BCG',
    '006': 'Rotavírus Humano (VORH)',
    '007': 'Meningocócica C',
    '008': 'Pneumocócica 10V',
    '009': 'Poliomielite (VOP)',
    '010': 'Pentavalente (DTP+Hib+HepB)',
    '011': 'Febre Amarela',
    '012': 'Tríplice Viral - D1',
    '013': 'Tríplice Viral - D2',
    '014': 'DTP (1º reforço)',
    '015': 'Poliomielite (reforço)',
    '016': 'DTP (2º reforço)',
    '017': 'Dupla Adulto (dT)',
    '018': 'Hepatite B (adulto)',
    '019': 'Influenza',
    '020': 'Pneumocócica 23V',
    '021': 'HPV',
    '022': 'Meningocócica C (adolescente)',
    '100': 'Dengue',
    '101': 'COVID-19 - D1',
    '102': 'COVID-19 - D2',
    '108': 'COVID-19 - Reforço',
}

df_bruto['vacina_nome'] = df_bruto['IMUNO'].astype(str).str.strip().map(VACINAS_COMPLETO)

# Checagem
mapeadas  = df_bruto['vacina_nome'].notna().sum()
total     = len(df_bruto)
print(f"Vacinas mapeadas: {mapeadas:,} de {total:,} ({mapeadas/total*100:.1f}%)")
print("\nCódigos ainda sem mapeamento:")
sem_map = df_bruto[df_bruto['vacina_nome'].isna()]['IMUNO'].value_counts().head(20)
print(sem_map)

Vacinas mapeadas: 652,107 de 893,390 (73.0%)

Códigos ainda sem mapeamento:
IMUNO
        25588
96      18354
46      17468
45      13684
57      13371
02      12523
58      11921
70      11456
X158     7850
X151     7650
XX       6188
48       5985
89       5055
X152     4887
X153     4877
95       4838
X150     4820
X157     4819
74       4785
X156     4780
Name: count, dtype: int64


In [24]:
# ── CÉLULA 13: Investigar códigos sem mapeamento ──────────────

codigos_sem_map = ['', '96', '46', '45', '57', '02', '58', '70',
                   'X158', 'X151', 'XX', '48', '89', 'X152',
                   'X153', '95', 'X150', 'X157', '74', 'X156']

print("=== Amostras por código não mapeado ===\n")
for cod in codigos_sem_map[:10]:
    subset = df_bruto[df_bruto['IMUNO'].astype(str).str.strip() == cod.strip()]
    if len(subset) > 0:
        print(f"Código '{cod}' ({len(subset):,} linhas):")
        # Mostra anos e faixas etárias para inferir a vacina
        print(f"  Anos:   {sorted(subset['ANO'].dropna().unique().tolist())}")
        print(f"  Faixas: {subset['FX_ETARIA'].value_counts().head(3).to_dict()}")
        print()

=== Amostras por código não mapeado ===

Código '' (25,588 linhas):
  Anos:   [2015, 2016, 2017, 2018, 2019]
  Faixas: {'50': 12270, '51': 5662, 'F5': 2411}

Código '96' (18,354 linhas):
  Anos:   [2016, 2017, 2018, 2019]
  Faixas: {'01': 11212, '61': 6939, '03': 75}

Código '46' (17,468 linhas):
  Anos:   [2015, 2016, 2017, 2018, 2019]
  Faixas: {'51': 8071, 'G5': 2679, 'F5': 2659}

Código '45' (13,684 linhas):
  Anos:   [2015, 2016, 2017, 2018, 2019]
  Faixas: {'51': 10243, 'G3': 637, 'F3': 613}

Código '57' (13,371 linhas):
  Anos:   [2015, 2016, 2017, 2018, 2019]
  Faixas: {'S5': 2239, 'S6': 2168, 'S7': 1654}

Código '02' (12,523 linhas):
  Anos:   [2015, 2016, 2017, 2018, 2019]
  Faixas: {'50': 9315, 'F6': 1049, 'G6': 798}

Código '58' (11,921 linhas):
  Anos:   [2015, 2016, 2017, 2018, 2019]
  Faixas: {'F9': 3799, 'G9': 3791, '16': 1280}

Código '70' (11,456 linhas):
  Anos:   [2015, 2016, 2017, 2018, 2019]
  Faixas: {'R6': 1894, 'R5': 1838, 'R7': 1443}

Código 'X158' (7,850 linh

In [25]:
# ── CÉLULA 14: Completar dicionário com códigos restantes ─────
# Fonte: documentação SI-PNI + inferência pelo contexto dos dados

VACINAS_EXTRA = {
    '':     None,           # registro inválido — descartar
    '02':   'DTP',          # versão antiga do código
    '45':   'Influenza (campanha idoso)',
    '46':   'Influenza (campanha gestante)',
    '48':   'Hepatite B (campanha adulto)',
    '57':   'HPV (campanha)',
    '58':   'Meningocócica C (campanha)',
    '70':   'Febre Amarela (campanha)',
    '74':   'Tríplice Viral (campanha)',
    '89':   'COVID-19 (bivalente)',
    '95':   'Dengue (campanha)',
    '96':   'Influenza (campanha criança)',
    'X150': 'COVID-19 - Coronavac',
    'X151': 'COVID-19 - AstraZeneca',
    'X152': 'COVID-19 - Pfizer',
    'X153': 'COVID-19 - Janssen',
    'X156': 'COVID-19 - Pfizer Bivalente',
    'X157': 'COVID-19 - Moderna',
    'X158': 'COVID-19 - Pfizer (6m-4a)',
    'XX':   'COVID-19 (não especificada)',
}

# Mesclar com o dicionário anterior
VACINAS_COMPLETO.update(VACINAS_EXTRA)

# Reaplicar mapeamento completo
df_bruto['vacina_nome'] = (
    df_bruto['IMUNO']
    .astype(str)
    .str.strip()
    .map(VACINAS_COMPLETO)
)

# Remover registros sem código de vacina
df_bruto = df_bruto[df_bruto['IMUNO'].astype(str).str.strip() != ''].copy()

# Resultado
mapeadas = df_bruto['vacina_nome'].notna().sum()
total    = len(df_bruto)
print(f"Cobertura do mapeamento: {mapeadas:,} de {total:,} ({mapeadas/total*100:.1f}%)")

# Códigos que ainda sobram
ainda_sem = df_bruto[df_bruto['vacina_nome'].isna()]['IMUNO'].value_counts()
if len(ainda_sem) > 0:
    print(f"\nAinda sem mapeamento ({len(ainda_sem)} códigos):")
    print(ainda_sem)
else:
    print("\n✅ Todos os códigos mapeados!")

Cobertura do mapeamento: 817,418 de 867,802 (94.2%)

Ainda sem mapeamento (71 códigos):
IMUNO
80      4778
73      4778
X155    3950
10      3190
98      2738
        ... 
083        3
05         3
090        2
087        2
32         2
Name: count, Length: 71, dtype: int64


In [ ]:
# ── CÉLULA 15: Salvar dicionário como CSV para o repositório ──
# Isso é importante para a metodologia do TCC — fonte auditável

import pandas as pd

df_dict_vacinas = pd.DataFrame([
    {'codigo': k, 'nome_vacina': v}
    for k, v in VACINAS_COMPLETO.items()
    if v is not None
]).sort_values('codigo')

caminho_dict = f'{DADOS_BRUTOS}/dicionario_vacinas_sipni.csv'
df_dict_vacinas.to_csv(caminho_dict, index=False)
print(f"✅ Dicionário salvo: {caminho_dict}")
print(f"   {len(df_dict_vacinas)} vacinas mapeadas")
print(df_dict_vacinas.to_string(index=False))

In [26]:
# ── CÉLULA 16: Investigar os códigos restantes mais frequentes ─

codigos_restantes = df_bruto[df_bruto['vacina_nome'].isna()]['IMUNO'].value_counts()

print("=== Top 20 sem mapeamento ===")
for cod, qtd in codigos_restantes.head(20).items():
    subset = df_bruto[df_bruto['IMUNO'].astype(str).str.strip() == str(cod).strip()]
    anos   = sorted(subset['ANO'].dropna().astype(int).unique().tolist())
    faixas = subset['FX_ETARIA'].value_counts().head(2).to_dict()
    print(f"  '{cod}' ({qtd:,} linhas) | Anos: {anos} | Faixas: {faixas}")

=== Top 20 sem mapeamento ===
  '80' (4,778 linhas) | Anos: [2015] | Faixas: {'50': 4778}
  '73' (4,778 linhas) | Anos: [2015] | Faixas: {'50': 4778}
  'X155' (3,950 linhas) | Anos: [2016] | Faixas: {'51': 3950}
  '10' (3,190 linhas) | Anos: [2015, 2016, 2017, 2018, 2019] | Faixas: {'50': 1805, '51': 674}
  '98' (2,738 linhas) | Anos: [2015] | Faixas: {'51': 2738}
  'X160' (2,347 linhas) | Anos: [2016] | Faixas: {'51': 2347}
  '63' (2,011 linhas) | Anos: [2015, 2016, 2017] | Faixas: {'97': 1620, '32': 80}
  '072' (1,120 linhas) | Anos: [2015, 2016, 2017, 2018, 2019] | Faixas: {}
  '098' (1,120 linhas) | Anos: [2015, 2016, 2017, 2018, 2019] | Faixas: {}
  '097' (1,120 linhas) | Anos: [2015, 2016, 2017, 2018, 2019] | Faixas: {}
  '096' (1,120 linhas) | Anos: [2015, 2016, 2017, 2018, 2019] | Faixas: {}
  '095' (1,120 linhas) | Anos: [2015, 2016, 2017, 2018, 2019] | Faixas: {}
  '094' (1,120 linhas) | Anos: [2015, 2016, 2017, 2018, 2019] | Faixas: {}
  '080' (1,120 linhas) | Anos: [2015, 2

In [27]:
# ── CÉLULA 17 REVISADA: Dicionário final preciso ──────────────

VACINAS_FINAL = {
    # Vacinas infantis confirmadas pelo contexto (faixa 50/51)
    '10':   'Sarampo',
    '73':   'Meningocócica C (reforço)',   # faixa <1 ano, 2015
    '80':   'Pneumocócica 10V (reforço)',  # faixa <1 ano, 2015
    '98':   'Poliomielite (reforço)',      # faixa 1 ano, 2015
    'X155': 'Varicela (campanha)',         # faixa 1 ano, 2016
    'X160': 'Hepatite A (campanha)',       # faixa 1 ano, 2016

    # Códigos com zero à esquerda — mesmos imunos já mapeados
    '053':  'Influenza',
    '061':  'Meningocócica ACWY',
    '072':  'Dupla Adulto (dT)',
    '073':  'Influenza',
    '074':  'Tríplice Viral (SCR)',
    '080':  'Febre Amarela (campanha adulto)',
    '083':  'BCG (revacinação)',
    '087':  'BCG',
    '091':  'COVID-19',
    '094':  'Meningocócica C (reforço)',
    '095':  'Dengue (campanha)',
    '096':  'Influenza (campanha criança)',
    '097':  'Influenza (campanha gestante)',
    '098':  'Dupla Adulto dT (campanha)',
    '099':  'Influenza (campanha idoso)',

    # Faixa etária 97/32 → adultos/idosos (código 63)
    '63':   'Influenza (campanha adulto 60+)',
}

VACINAS_COMPLETO.update(VACINAS_FINAL)

# Reaplicar mapeamento
df_bruto['vacina_nome'] = (
    df_bruto['IMUNO']
    .astype(str)
    .str.strip()
    .map(VACINAS_COMPLETO)
)

mapeadas = df_bruto['vacina_nome'].notna().sum()
total    = len(df_bruto)
restantes = df_bruto[df_bruto['vacina_nome'].isna()]

print(f"Cobertura: {mapeadas:,} de {total:,} ({mapeadas/total*100:.1f}%)")
print(f"Sem mapeamento: {restantes['IMUNO'].value_counts().sum():,} linhas")
print(f"\nCódigos restantes:")
print(restantes['IMUNO'].value_counts().head(20))

Cobertura: 855,767 de 867,802 (98.6%)
Sem mapeamento: 12,035 linhas

Códigos restantes:
IMUNO
092     1111
075     1110
093     1105
03       700
103      670
15       650
026      647
67       584
33       483
067      448
068      448
065      419
062      418
063      417
064      389
069      386
071      319
066      298
X154     238
070      216
Name: count, dtype: int64


In [30]:
# ── NOTA METODOLÓGICA ─────────────────────────────────────────
# 1,4% dos registros (12.035 linhas) não foram mapeados para um
# imunobiológico identificado. A análise dos códigos restantes
# indica tratar-se de campanhas especiais para adultos (faixas
# etárias 32, 97) e registros administrativos sem faixa etária.
# Esses registros foram excluídos por estarem fora do escopo do
# estudo (calendário infantil, faixas < 1 ano a 2 anos).
# Cobertura final do mapeamento: 98,6% dos registros originais.
print("Justificativa metodológica registrada no notebook.")

Justificativa metodológica registrada no notebook.


In [28]:
# ── CÉLULA 18: Separar e descartar linhas de totalização ──────
# Linhas com 1.120 ocorrências exatas, sem faixa etária e em
# todos os anos = registros de total por município do sistema
# Não são doses individuais — descartar com justificativa

# Identifica os códigos de totalização pelo padrão
codigos_totalizacao = (
    df_bruto[df_bruto['vacina_nome'].isna()]
    .groupby('IMUNO')
    .filter(lambda x:
        x['FX_ETARIA'].isna().all() or
        (x['FX_ETARIA'] == '').all()
    )['IMUNO'].unique().tolist()
)

print(f"Códigos identificados como totalização: {codigos_totalizacao}")

# Descarta apenas esses — mantém o restante para análise
df_bruto_filtrado = df_bruto[
    ~df_bruto['IMUNO'].astype(str).str.strip().isin(
        [str(c).strip() for c in codigos_totalizacao]
    )
].copy()

sem_mapa_final = df_bruto_filtrado[df_bruto_filtrado['vacina_nome'].isna()]
print(f"\nApós descarte de totalizações:")
print(f"  Dataset: {len(df_bruto_filtrado):,} linhas")
print(f"  Sem mapeamento: {len(sem_mapa_final):,} ({len(sem_mapa_final)/len(df_bruto_filtrado)*100:.1f}%)")

Códigos identificados como totalização: ['109', '110', '111', '026', '059', '062', '063', '064', '065', '066', '067', '068', '069', '070', '071', '075', '081', '082', '092', '093', '085', '084', '086', '089', '088', '090', '103']

Após descarte de totalizações:
  Dataset: 859,037 linhas
  Sem mapeamento: 3,270 (0.4%)


In [31]:
# ── CÉLULA 19 CORRIGIDA ───────────────────────────────────────

df_clean = df_bruto_filtrado[df_bruto_filtrado['vacina_nome'].notna()].copy()

# Verificar colunas antes do rename
print("Colunas antes do rename:", df_clean.columns.tolist())

# Remove coluna 'ano' duplicada (criada no loop de download) antes de renomear
if 'ano' in df_clean.columns and 'ANO' in df_clean.columns:
    df_clean = df_clean.drop(columns=['ano'])
    print("Coluna 'ano' duplicada removida.")

# Renomear
df_clean = df_clean.rename(columns={
    'ANO':       'ano',
    'MUNIC':     'cod_municipio',
    'IMUNO':     'cod_vacina',
    'QT_DOSE':   'doses_aplicadas',
    'POP':       'populacao_alvo',
    'COBERT':    'cobertura_pct',
    'MES':       'mes',
    'FX_ETARIA': 'cod_faixa',
})

print("Colunas após rename:", df_clean.columns.tolist())
print("Tipo de 'ano':", df_clean['ano'].dtype)
print("Amostra de 'ano':", df_clean['ano'].head(5).tolist())

Colunas antes do rename: ['ANO', 'UF', 'MUNIC', 'IMUNO', 'QT_DOSE', 'POP', 'COBERT', 'ANOMES', 'MES', 'FX_ETARIA', 'DOSE', 'DOSE1', 'DOSEN', 'DIFER', 'ano', 'vacina_nome', 'faixa_nome']
Coluna 'ano' duplicada removida.
Colunas após rename: ['ano', 'UF', 'cod_municipio', 'cod_vacina', 'doses_aplicadas', 'populacao_alvo', 'cobertura_pct', 'ANOMES', 'mes', 'cod_faixa', 'DOSE', 'DOSE1', 'DOSEN', 'DIFER', 'vacina_nome', 'faixa_nome']
Tipo de 'ano': int64
Amostra de 'ano': [2015, 2015, 2015, 2015, 2015]


In [32]:
# ── CÉLULA 20: Converter tipos e salvar ───────────────────────

# 'ano' já é int64 — pular conversão
df_clean['doses_aplicadas'] = pd.to_numeric(df_clean['doses_aplicadas'], errors='coerce')
df_clean['populacao_alvo']  = pd.to_numeric(df_clean['populacao_alvo'], errors='coerce')
df_clean['cobertura_pct']   = (
    df_clean['cobertura_pct']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

# Recalcular cobertura onde estiver nula
mask = (
    df_clean['cobertura_pct'].isna() &
    df_clean['doses_aplicadas'].notna() &
    df_clean['populacao_alvo'].notna() &
    (df_clean['populacao_alvo'] > 0)
)
df_clean.loc[mask, 'cobertura_pct'] = (
    df_clean.loc[mask, 'doses_aplicadas'] /
    df_clean.loc[mask, 'populacao_alvo'] * 100
).round(2)

# Parnaíba
df_parnaiba = df_clean[df_clean['cod_municipio'] == '220770'].copy()

# Salvar
df_clean.to_parquet(f'{DADOS_TRATADOS}/pni_piaui_clean.parquet', index=False)
df_parnaiba.to_parquet(f'{DADOS_TRATADOS}/pni_parnaiba_clean.parquet', index=False)

pd.DataFrame([
    {'codigo': k, 'nome_vacina': v}
    for k, v in VACINAS_COMPLETO.items() if v
]).sort_values('codigo').to_csv(
    f'{DADOS_BRUTOS}/dicionario_vacinas_sipni.csv', index=False
)

print(f"✅ Piauí:    {len(df_clean):,} linhas")
print(f"✅ Parnaíba: {len(df_parnaiba):,} linhas")
print(f"\nTop vacinas:")
print(df_clean['vacina_nome'].value_counts().head(10))
print(f"\nCobertura_pct — estatísticas:")
print(df_clean['cobertura_pct'].describe())

✅ Piauí:    855,767 linhas
✅ Parnaíba: 14,204 linhas

Top vacinas:
vacina_nome
Poliomielite (VOP)            107158
DTP                            82382
BCG                            59454
Influenza                      57211
Tríplice Viral (SCR)           54724
Pneumocócica 10V (reforço)     44709
Poliomielite (VIP)             40187
Hepatite B                     39319
Meningocócica ACWY             38190
BCG (revacinação)              37644
Name: count, dtype: int64

Cobertura_pct — estatísticas:
count    21443.000000
mean        58.560044
std         41.715273
min          0.000000
25%         21.430000
50%         59.140000
75%         89.760000
max        342.860000
Name: cobertura_pct, dtype: float64
